# Assignment 1 -- ML4F 2023

## Instructions
* This assignment covers the material discussed in Lectures ML1 - ML4. 
* Each group submits _only one_ notebook via canvas on the assignment page. 
* The notebook should be named `assignment1_groupXX.ipynb` where `XX` is your group number,  
e.g. for group 3 this will be `assignment1_group03.ipynb`.
* The notebook should run without raising any errors. 
* We recommend keeping the folder structure
```
assignment/
    data/
    lib/
    assignment1_groupXX.ipynb
```
* We strongly recommend git, as you are encouraged to collaborate and split up the work and maybe even start independently. To see how to set up your own repo for your group, see `VU Workshop Introduction to Version Control with GIT.pptx` discussed in week 2.
* Do not spend time on optimizing the speed of your code. However, if it runs for more than 5 minutes, we will terminate it.
* We strongly encourage you to experiment, try different approaches and combinations and get to know the problem from alternative angles. But the final notebook should only contain the necessary results for grading.
----

<div style="font-size:24px; text-align:center; font-weight:bold">Good luck!</div>

----

# Assignment 1 - Features & Algorithms

Kiwibank, a commercial bank from New Zealand, is interested in updating their loan default prediction algorithm. They used to check each client manually, and now they want to use Machine Learning to predict who will default on their loan and pose a threat to the bank's balance sheet. They have supplied you with a dataset of their past clients, and they've asked you to consult them on their adaptation of Machine Learning in this process.

First take a look at the data, then test different algorithms, select key features and write a recommendation to the bank.

State your imports below.

In [201]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn.preprocessing import normalize
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score,matthews_corrcoef,classification_report,roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import ElasticNet

# Preprocessing (15 points)
*5 points for correct code*

*10 points for normalization procedure*

The data consists of 34 features and one target variable, 'Loan Status'
We have 25 numerical features and 9 categorical features. For the numerical features, apply a technique
to make the data more normal. This can be Standardization, Normalization, log or a Box-Cox transformation. 
Explain why you might use one method for some features, and another for other features.
Also check for missing values and uninformative features (NO FEATURE SELECTION YET).

In [41]:
data = pd.read_csv('8_ML4F_Assignment1Data.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 35 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ID                            10000 non-null  int64  
 1   Loan Amount                   10000 non-null  int64  
 2   Funded Amount                 10000 non-null  int64  
 3   Funded Amount Investor        10000 non-null  float64
 4   Term                          10000 non-null  int64  
 5   Batch Enrolled                10000 non-null  object 
 6   Interest Rate                 10000 non-null  float64
 7   Grade                         10000 non-null  object 
 8   Sub Grade                     10000 non-null  object 
 9   Employment Duration           10000 non-null  object 
 10  Home Ownership                10000 non-null  float64
 11  Verification Status           10000 non-null  object 
 12  Payment Plan                  10000 non-null  object 
 13  Lo

In [60]:
#data_pp1 = preprocessing.normalize(data_pp, norm='l2')
#df_normalized = pd.DataFrame(data_pp1)
#df_normalized.columns = data_pp.columns

In [103]:
data_n = data.loc[:,['Loan Amount','Funded Amount','Funded Amount Investor','Term','Interest Rate','Home Ownership','Debit to Income','Delinquency - two years','Inquires - six months','Open Account','Public Record','Revolving Balance','Revolving Utilities','Total Accounts','Total Received Interest','Total Received Late Fee','Recoveries','Collection Recovery Fee','Last week Pay','Total Collection Amount','Total Current Balance','Total Revolving Credit Limit']]
data_c = data.loc[:,['Grade','Sub Grade','Employment Duration','Verification Status','Payment Plan','Loan Title','Initial List Status','Application Type']]

In [105]:
#Encode categorical data
label_encoder = LabelEncoder()
data_c['Grade_l'] = label_encoder.fit_transform(data_c['Grade'])
data_c['Employment_Duration_l'] = label_encoder.fit_transform(data_c['Employment Duration'])
data_c['Verification_Status_l'] = label_encoder.fit_transform(data_c['Verification Status'])
data_c['Payment_Plan_l'] = label_encoder.fit_transform(data_c['Payment Plan'])
data_c['Loan_Title_l'] = label_encoder.fit_transform(data_c['Loan Title'])
data_c['Initial_List_Status_l'] = label_encoder.fit_transform(data_c['Initial List Status'])
data_c['Application_Type_l'] = label_encoder.fit_transform(data_c['Application Type'])
data_c['Sub_Grade_l1'] = data_c['Sub Grade'].str[1]
data_c['Sub_Grade_l2'] = data_c['Grade_l'].astype(str) + data_c['Sub_Grade_l1'].astype(str)
data_c_encode = data_c.loc[:,['Grade_l','Employment_Duration_l','Verification_Status_l','Payment_Plan_l','Loan_Title_l','Initial_List_Status_l','Application_Type_l','Sub_Grade_l2']]

In [66]:
#Standardized numerical data
data_np = StandardScaler().fit_transform(data_n.values)
data_standardized = pd.DataFrame(data_np)
data_standardized.columns = data_pp.columns
data_standardized

,Loan Amount,Funded Amount,Funded Amount Investor,Term,Interest Rate,Home Ownership,Debit to Income,Delinquency - two years,Inquires - six months,Open Account,...,Revolving Utilities,Total Accounts,Total Received Interest,Total Received Late Fee,Recoveries,Collection Recovery Fee,Last week Pay,Total Collection Amount,Total Current Balance,Total Revolving Credit Limit
0,-0.103757,0.149881,-0.571898,0.248853,0.179244,1.064635,-1.360944,-0.404075,-0.302791,-0.055311,...,1.136524,-0.545010,-0.788428,-0.207180,-0.154205,-0.281321,-0.489957,-0.142053,-0.787155,-0.780309
1,1.158060,-1.350154,0.035964,-0.039706,0.802482,1.806642,0.335029,-0.404075,-0.302791,-0.055311,...,-0.274894,-0.545010,-0.408131,-0.200401,-0.140536,-0.239479,-1.322152,-0.142053,1.067000,-0.661471
2,-0.602668,2.146082,-0.052201,-0.039706,1.868081,1.570170,-0.769888,0.799604,-0.302791,-0.055311,...,-2.045434,0.802357,0.081010,-0.204619,-0.159044,0.096450,-0.143209,-0.167660,0.174348,-0.955854
3,-0.026882,-1.082742,-1.095386,-0.039706,0.928007,1.303618,0.816143,-0.404075,-0.302791,-0.055311,...,-1.380041,0.679869,-0.694532,-0.188284,-0.145618,-0.160592,1.775463,-0.133518,0.276107,-0.271175
4,1.984255,-0.332909,-1.204406,-0.039706,-0.174250,-0.581854,1.576513,-0.404075,-0.302791,-0.536278,...,1.434046,0.067430,0.242686,-0.213048,-0.159981,0.079673,0.966384,-0.139208,-0.148998,-0.731417
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.745321,-0.359785,-0.710374,-0.039706,0.211298,-0.496936,-0.639395,-0.404075,-0.302791,0.105011,...,0.565240,-0.055058,-0.711163,-0.200072,-0.166488,-0.089231,-1.160337,-0.119292,3.412060,-0.612111
9996,0.087417,-0.067583,-0.121342,0.248853,-0.580393,0.970693,0.575885,-0.404075,-0.302791,1.227268,...,-0.063136,-0.667498,0.995466,-0.201767,-0.168133,-0.310095,-1.299036,-0.154857,1.973778,0.014505
9997,-0.250236,2.058090,-0.170973,0.248853,2.603930,-0.167468,-0.257244,-0.404075,1.841619,-1.017245,...,-1.952960,-1.157450,0.296093,-0.210124,-0.168110,-0.023749,-0.143209,-0.181885,0.056202,0.264768
9998,1.090125,-0.886018,-0.366263,-0.039706,0.048211,-0.273152,-0.975154,-0.404075,-0.302791,0.585978,...,-0.309871,0.679869,-0.656287,-0.207158,-0.152695,-0.117027,1.798580,-0.144899,2.126739,-0.510724


In [113]:
df = pd.merge(data_c_encode, data_standardized, left_index=True, right_index=True)
df['Collection 12 months Medical'] = data['Collection 12 months Medical']
df['Loan Status'] = data['Loan Status']
df

,Grade_l,Employment_Duration_l,Verification_Status_l,Payment_Plan_l,Loan_Title_l,Initial_List_Status_l,Application_Type_l,Sub_Grade_l2,Loan Amount,Funded Amount,...,Total Received Interest,Total Received Late Fee,Recoveries,Collection Recovery Fee,Last week Pay,Total Collection Amount,Total Current Balance,Total Revolving Credit Limit,Collection 12 months Medical,Loan Status
0,0,2,1,0,46,1,0,01,-0.103757,0.149881,...,-0.788428,-0.207180,-0.154205,-0.281321,-0.489957,-0.142053,-0.787155,-0.780309,0,0
1,1,0,1,0,36,0,0,12,1.158060,-1.350154,...,-0.408131,-0.200401,-0.140536,-0.239479,-1.322152,-0.142053,1.067000,-0.661471,0,0
2,0,0,1,0,46,1,0,04,-0.602668,2.146082,...,0.081010,-0.204619,-0.159044,0.096450,-0.143209,-0.167660,0.174348,-0.955854,0,0
3,0,0,0,0,53,1,0,03,-0.026882,-1.082742,...,-0.694532,-0.188284,-0.145618,-0.160592,1.775463,-0.133518,0.276107,-0.271175,0,1
4,4,2,0,0,36,1,0,41,1.984255,-0.332909,...,0.242686,-0.213048,-0.159981,0.079673,0.966384,-0.139208,-0.148998,-0.731417,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2,2,2,0,36,1,0,21,0.745321,-0.359785,...,-0.711163,-0.200072,-0.166488,-0.089231,-1.160337,-0.119292,3.412060,-0.612111,0,0
9996,1,0,0,0,36,0,0,12,0.087417,-0.067583,...,0.995466,-0.201767,-0.168133,-0.310095,-1.299036,-0.154857,1.973778,0.014505,0,0
9997,2,2,2,0,36,1,0,22,-0.250236,2.058090,...,0.296093,-0.210124,-0.168110,-0.023749,-0.143209,-0.181885,0.056202,0.264768,0,0
9998,4,1,1,0,36,1,0,43,1.090125,-0.886018,...,-0.656287,-0.207158,-0.152695,-0.117027,1.798580,-0.144899,2.126739,-0.510724,0,0


# Training (30 points)
*10 points for correct code*

*10 points for interpretation results*

*10 points for pitfalls*

We've familiarized ourselves with the data, so now we're going to train some models.
A handful of frequently used models are: Decision Trees (DTC), Nearest Neighbors (KNN) and
Stochastic Gradient Descent learning algorithm (SGD).
Split the data into a training set and a test set, and train all mentioned models on the data.
For KNN, find the optimal number of neighbors. 
Use the test set to get the predicted values. Show the performance of the different models in a Confusion Matrix and compare the accuracy, precision, recall and F1-score.

When looking at the resulting measures and matrices, what stands out? What possible pitfalls may be hiding in each of the models?

In [130]:
features = df.columns[:-1]
X = df[features]
Y = df['Loan Status']
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1)

In [156]:
model1 = DecisionTreeClassifier()
model1 = model1.fit(X_train,y_train)
y_pred1 = model1.predict(X_test)

In [159]:
matrix1 = pd.DataFrame(confusion_matrix(y_test, y_pred1))
matrix1.columns = ('Predicted 0', 'Predicted 1')
matrix1 = matrix1.rename (index = {0: 'Actual 0', 1:'Actual 1'})
matrix1

,Predicted 0,Predicted 1
Actual 0,2443,288
Actual 1,229,40


In [172]:
model2 = KNeighborsClassifier(n_neighbors=5)
model2.fit(X_train, y_train)
y_pred2 = model2.predict(X_test)

In [173]:
matrix2 = pd.DataFrame(confusion_matrix(y_test, y_pred2))
matrix2.columns = ('Predicted 0', 'Predicted 1')
matrix2 = matrix2.rename (index = {0: 'Actual 0', 1:'Actual 1'})
matrix2

,Predicted 0,Predicted 1
Actual 0,2709,22
Actual 1,269,0


In [197]:
model3 = SGDClassifier(loss='log_loss', max_iter=1000, random_state=0)
model3.fit(X_train, y_train)
y_pred3 = model3.predict(X_test)

In [198]:
matrix3= pd.DataFrame(confusion_matrix(y_test, y_pred3))
matrix3.columns = ('Predicted 0', 'Predicted 1')
matrix3 = matrix3.rename (index = {0: 'Actual 0', 1:'Actual 1'})
matrix3

,Predicted 0,Predicted 1
Actual 0,2728,3
Actual 1,268,1


In [168]:
accuracy_1 = accuracy_score(y_test, y_pred1)
precision_1 = precision_score(y_test, y_pred1)
recall_1 = recall_score(y_test, y_pred1)
f1_1 = f1_score(y_test, y_pred1)
print("Model 1:")
print("Accuracy:", accuracy_1)
print("Precision:", precision_1)
print("Recall:", recall_1)
print("F1-score:", f1_1)
print()

Model 1:
Accuracy: 0.8276666666666667
Precision: 0.12195121951219512
Recall: 0.14869888475836432
F1-score: 0.13400335008375208



In [170]:
accuracy_2 = accuracy_score(y_test, y_pred2)
precision_2 = precision_score(y_test, y_pred2)
recall_2 = recall_score(y_test, y_pred2)
f1_2 = f1_score(y_test, y_pred2)
print("Model 2:")
print("Accuracy:", accuracy_2)
print("Precision:", precision_2)
print("Recall:", recall_2)
print("F1-score:", f1_2)
print()

Model 2:
Accuracy: 0.889
Precision: 0.05555555555555555
Recall: 0.01486988847583643
F1-score: 0.023460410557184754



In [199]:
accuracy_3 = accuracy_score(y_test, y_pred3)
precision_3 = precision_score(y_test, y_pred3)
recall_3 = recall_score(y_test, y_pred3)
f1_3 = f1_score(y_test, y_pred3)
print("Model 3:")
print("Accuracy:", accuracy_3)
print("Precision:", precision_3)
print("Recall:", recall_3)
print("F1-score:", f1_3)
print()

Model 3:
Accuracy: 0.9096666666666666
Precision: 0.25
Recall: 0.0037174721189591076
F1-score: 0.007326007326007325



# Feature Selection (35 points)
*10 points for correct code*

*15 points for correct reasoning and interpretation*

*10 points for explanation pitfalls*

Now that we've compared the performance of the different models, we place our judgement upon 
the dimensionality of the data. The dimension reduction methods discussed so far are 
L2 regularization and LASSO. Combining these methods gives us the Elastic Net method. What are some pitfalls of L2 regularization and of LASSO? How does Elastic Net overcome these pitfalls?

The Elastic Net method uses two parameters, l1_ratio and α. l1_ratio takes on a value between 0 and 1, and α a value higher than 0. Use the Elastic Net method to get an impression of the importance of the features, and make 
an appropriate and argumented decision regarding their individual inclusion.

In [243]:
model4 = ElasticNet(alpha=0.005, l1_ratio=0.5, random_state=0)
model4.fit(X_train, y_train)

ElasticNet(alpha=0.005, random_state=0)

In [244]:
feature_coef = pd.Series(model4.coef_, index=X.columns)
feature_coef

Grade_l                        -0.000000
Employment_Duration_l           0.000000
Verification_Status_l           0.000000
Payment_Plan_l                  0.000000
Loan_Title_l                    0.000097
Initial_List_Status_l          -0.007111
Application_Type_l              0.000000
Sub_Grade_l2                    0.000087
Loan Amount                    -0.000896
Funded Amount                   0.000000
Funded Amount Investor         -0.000000
Term                            0.000000
Interest Rate                  -0.000000
Home Ownership                  0.000000
Debit to Income                -0.003735
Delinquency - two years        -0.000247
Inquires - six months          -0.000000
Open Account                   -0.001678
Public Record                   0.000000
Revolving Balance              -0.000000
Revolving Utilities            -0.004508
Total Accounts                 -0.003257
Total Received Interest        -0.000000
Total Received Late Fee         0.002922
Recoveries      

In [245]:
selected_features = feature_coef[feature_coef != 0].index
selected_features

Index(['Loan_Title_l', 'Initial_List_Status_l', 'Sub_Grade_l2', 'Loan Amount',
       'Debit to Income', 'Delinquency - two years', 'Open Account',
       'Revolving Utilities', 'Total Accounts', 'Total Received Late Fee',
       'Collection Recovery Fee', 'Total Collection Amount',
       'Total Revolving Credit Limit'],
      dtype='object')

# Recommendation (10 points)
*10 points for paper*

Now that you've made an assessment of the key features and it is clear which model performs best, it is time to write a recommendation to Kiwibank. Explain what you have researched and present your results in a short paper in no more than 400 words. 

Focus on the bank's wants and needs, and minimize the technical talk. We recommend writing in LaTeX/Overleaf, but Word or another application is also fine. Hand the paper in as PDF, together with your Jupyter Notebook, in a ZIP-file.